[Guide Index](README.md)

# Appendix D: How StarLayer classifies errors

When something goes wrong, StarLayer raises an error from one small family of classes. This appendix explains the family in plain terms: how to tell which error you will get, how to catch it, and what changed from the earlier release. It ends with the open questions about the scheme, for review.

The whole idea fits in three rules:

1. **A wrong argument is a `ValueError` or a `TypeError`.** You called a function with something it refuses. Change the argument and the error goes away.
2. **Anything else StarLayer raises on purpose is one of four classes, and the class says what kind of problem it is.** The text or data is malformed (`FormatError`), the shapes are wrong (`ShapeError`), the work could not be done (`ExecutionError`), or a SPARQL endpoint failed (`EndpointError`).
3. **The class names the cause, and a `kind` says which cause.** There is one `FormatError`, not one class per language, and its `kind` says whether the malformed thing was SPARQL, SRL, RDF, a term, Manchester Syntax or OWL. Where it happened (the function, the document) is the `context`, never a different class.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom. The setup cell defines `show()`, which runs a call that is expected to fail and prints the error's class, its `kind`, its `context` and its message.

In [ ]:
from starlayer.errors import ExecutionError, FormatError, ShapeError, StarLayerError, only_kind
from starlayer.graph import XSD, StarLayerGraph, literal, parse_term, to_python
from starlayer.shacl import meta_validate
from starlayer.sparql import sparql_to_tree, srl, tree_to_sparql
from starlayer.starontology.manchester import manchester_to_tree


def show(label, call):
    """Run a call that is expected to fail, and print what was raised."""
    try:
        call()
    except Exception as error:
        kind = getattr(error, "kind", None)
        position = f" line {error.line}" if getattr(error, "line", None) else ""
        context = getattr(error, "context", None)
        print(f"{label}")
        print(f"   {type(error).__name__}, kind={kind!r}, context={context!r}{position}")
        message = getattr(error, "why", None) or str(error).splitlines()[0]
        print(f"   {message[:100]}")
    else:
        print(label, "-> no error")


g = StarLayerGraph()
g.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p ex:b .", format="turtle")

<StarLayerGraph DEFAULT (1 triples)>

## 1. The family

```
StarLayerError                  everything StarLayer raises on purpose
 |- FormatError                 text or data is malformed             kind, context, line, col
 |- ExecutionError              the work could not be done            kind, context
 |- ShapeError                  a shapes graph is wrong               context
 |   |- ShapeCompileError       the engine cannot read or compile it
 |   `- MetaShaclError          the meta check found the problem      problems, report_text
 `- EndpointError               a SPARQL endpoint failed              status (also an OSError)
```

`except StarLayerError` catches all of them. The classes are not a `ValueError`, a `SyntaxError` or a `RuntimeError`, so `except ValueError` no longer catches a malformed document or a wrong shapes graph. (`EndpointError` is also an `OSError`, because a network failure is an I/O error.)

## 2. Which error will I get?

Ask what went wrong:

| What went wrong | What is raised | A quick test |
| --- | --- | --- |
| I passed a value, type or combination of arguments the call refuses | `ValueError` or `TypeError` | changing the argument fixes it |
| The text, document, tree or term is malformed | `FormatError` | the input itself is not valid |
| The input is valid but the work could not be done | `ExecutionError` | it ran, or tried to, and failed or reached a limit |
| The shapes graph itself is wrong | `ShapeError` (or `MetaShaclError`) | the shapes cannot be used |
| A SPARQL endpoint did not answer | `EndpointError` | the network or the server failed |
| An optional package is missing | `ImportError` | it says what to install |

The `kind` then says which cause, and the next sections list them with an example of each.

## 3. `FormatError`: the text or data is malformed

`FormatError` has a `kind` for the language or format that is malformed:

| `kind` | The malformed thing |
| --- | --- |
| `sparql` | SPARQL query or update text, an `sqe:` tree, a `VERSION` line |
| `srl` | SRL text, an `srl:` tree, a bad `DATA` block, a rule set that breaks §4.2 |
| `rdf` | an RDF document that cannot be parsed, a list that is not well formed, tied or missing ordinals in a tree |
| `term` | the text of a term: an IRI, a language tag, a lexical form that does not fit its datatype |
| `manchester` | Manchester Syntax text, a `manch:` tree, a SWRL `Rule:` frame |
| `owl` | RDF that is not valid OWL when it is written as Manchester Syntax |

It carries `line` and `col` when the parser knows where the problem is, and `context` says which input it was.

Below, one malformed input of each of the most common kinds is read, and the class, kind, context and line are printed.

In [ ]:
show("SPARQL text that is not SPARQL", lambda: sparql_to_tree("this is not sparql"))
show("a query that contradicts its VERSION line", lambda: g.query('VERSION "1.1" SELECT ?s WHERE { ?s ?p <<( ?s ?p ?o )>> }'))
show("a Turtle document with a syntax error", lambda: StarLayerGraph().parse(data="<a> <b>", format="turtle"))
show("text that is not exactly one term", lambda: parse_term("4 5"))
show("a lexical form that does not fit its datatype", lambda: to_python(parse_term('"abc"^^xsd:integer', {"xsd": str(XSD)})))
show("an unfinished SRL rule", lambda: srl.parse_ruleset("RULE { ?x"))
show("Manchester text with a syntax error", lambda: manchester_to_tree("Prefix: : <http://example.org/>\nClass: A SubClassOf:"))
show("a SWRL rule in Manchester text (not supported)", lambda: manchester_to_tree("Prefix: : <http://example.org/>\nRule: Person(?x)"))

SPARQL text that is not SPARQL
   FormatError, kind='sparql', context=None line 1
   expected SELECT, ASK, CONSTRUCT or DESCRIBE (found 'this')
a query that contradicts its VERSION line
   FormatError, kind='sparql', context='VERSION'
   SPARQL query declares VERSION '1.1' but uses a triple term, which '1.1' conformance excludes (SPARQL
a Turtle document with a syntax error
   FormatError, kind='rdf', context='Turtle document' line 1
   Parser error at line 1 between columns 1 and 4: No scheme found in an absolute IRI (line 1)
text that is not exactly one term
   FormatError, kind='term', context=None
   Parser error at line 1 column 43: 5 is not a valid subject or graph name (line 1)
a lexical form that does not fit its datatype
   FormatError, kind='term', context='xsd:integer literal'
   'abc' is not a valid integer
an unfinished SRL rule
   FormatError, kind='srl', context=None line 1
   expected a predicate (found end of input)
Manchester text with a syntax error
   FormatError, k

## 4. `ExecutionError`: the work could not be done

The input is valid, but running it failed or reached a limit. The `kind` says why:

| `kind` | What failed |
| --- | --- |
| `query` | Oxigraph could not evaluate a query or update whose text parsed (for example a custom function that was not passed) |
| `rule` | a rule set could not be run: it cannot be ordered (stratification), or a rule limit was reached |
| `import` | an SRL `IMPORTS` document could not be loaded, or the imports were never resolved |
| `nesting` | the input is nested too deeply to process |
| `unsupported` | valid input that StarLayer cannot represent, render or rewrite (for example a Manchester construct with no counterpart) |
| `limit` | a size or complexity limit of the library was reached |
| `timeout`, `inconsistent`, `inference` | OWL 2 DL reasoning: the time budget ran out, the data is inconsistent, or the reasoner could not run (see [02.07](02.07-owl-dl-reasoning.ipynb)) |
| `expression`, `recursion`, `constraint-query` | SHACL: a node expression cannot be evaluated, a shape reaches itself through `sh:not`, a `sh:sparql` query reported a failure (see [05.02](05.02-starlayer-shacl-engine.ipynb)) |
| `internal` | a state that should be impossible: a bug in StarLayer, to be reported |

Below, one example each of `query`, `nesting`, `rule`, `import` and `unsupported`.

In [ ]:
show("a custom function that was not passed", lambda: list(g.query("SELECT (<urn:fn:x>(?s) AS ?y) WHERE { ?s ?p ?o }")))
show("a query nested far too deeply", lambda: sparql_to_tree("SELECT ?x WHERE " + "{ " * 3000 + "?x ?p ?o" + " }" * 3000))
show("a rule set that cannot be ordered", lambda: srl.parse_ruleset("PREFIX ex: <http://example.org/> RULE { ?x ex:f ex:y } WHERE { ?x a ex:C . NOT { ?x ex:f ex:y } }").plan())
show("a rule set with imports that were never resolved", lambda: srl.parse_ruleset("IMPORTS <http://example.org/rules> RULE { ?x <urn:p> ?y } WHERE { ?x <urn:q> ?y }").infer(g))

labelled = StarLayerGraph()
labelled.parse(data='@prefix : <http://x/> . @prefix owl: <http://www.w3.org/2002/07/owl#> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> . :A a owl:Class ; rdfs:label "x"@ar--rtl .', format="turtle")
show("a graph that Manchester Syntax cannot write", lambda: labelled.serialize(format="manchester"))

a custom function that was not passed
   ExecutionError, kind='query', context='Oxigraph query'
   The custom function <urn:fn:x> is not supported
a query nested far too deeply
   ExecutionError, kind='nesting', context='SPARQL parse'
   SPARQL expression or pattern nested too deeply to parse
a rule set that cannot be ordered
   ExecutionError, kind='rule', context='stratification'
   SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule
a rule set with imports that were never resolved
   ExecutionError, kind='import', context='IMPORTS <http://example.org/rules>'
   SRL: this rule set has IMPORTS that are not resolved (<http://example.org/rules>). Pass imports_load
a graph that Manchester Syntax cannot write
   ExecutionError, kind='unsupported', context='Manchester serialize'
   cannot render <Literal value=x language=ar direction=rtl>: Manchester syntax has no base direction f


## 5. `ShapeError`: the shapes are wrong

`ShapeError` is about the shapes graph, not the data. It is raised when the shapes cannot be read or compiled, and `MetaShaclError` (a `ShapeError`) is raised by the meta check, with the problems it found in `problems` and the full report in `report_text`.

Below, a shapes graph with an invalid `sh:datatype` is checked.

In [ ]:
bad_shapes = StarLayerGraph()
bad_shapes.parse(data="@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix ex: <http://example.org/> . ex:S a sh:NodeShape ; sh:property [ sh:path ex:p ; sh:datatype 5 ] .", format="turtle")
try:
    meta_validate(bad_shapes)
except ShapeError as error:
    print(type(error).__name__)
    print("problems:", error.problems)

MetaShaclError
problems: ('the property shape of ex:S for ex:p, sh:datatype: violates sh:or',)


## 6. Wrong arguments stay `ValueError` and `TypeError`

A call that is given an argument it refuses raises a `ValueError` (a bad value or combination) or a `TypeError` (the wrong type). These are not part of the family, because the cause is in the call and not in the data.

Below, a literal with both a language and a datatype, a name for an entailment regime that does not exist, and a term that is not text.

In [ ]:
show("a literal with a language and a datatype", lambda: literal("5", XSD.integer, lang="sv"))
show("an entailment regime that does not exist", lambda: g.infer(entailment="nonsense"))
show("parse_term given a number", lambda: parse_term(5))
show("tree_to_sparql on a graph with no query", lambda: tree_to_sparql(StarLayerGraph()))

a literal with a language and a datatype
   ValueError, kind=None, context=None
   a literal has a language tag or a datatype, not both: lang='sv' with datatype <http://www.w3.org/200
an entailment regime that does not exist
   ValueError, kind=None, context=None
   Unsupported entailment regime(s) ['nonsense'] - expected a combination of ['http://www.w3.org/ns/ent
parse_term given a number
   TypeError, kind=None, context=None
   parse_term() needs a str, not int
tree_to_sparql on a graph with no query
   ValueError, kind=None, context=None
   the graph holds no root (no sqe:Query, sqe:UpdateRequest or srl:RuleSet): nothing to write


## 7. Catching errors

Catch the narrowest class that you can handle. `except StarLayerError` is the catch-all for the library. When a handler is for one `kind` of an `ExecutionError` or `FormatError`, call `only_kind(error, "kind")` first: it raises the error again if its kind is a different one, so a handler never swallows a cause it was not written for.

Below, three texts go through one function that handles a malformed text, a text that is nested too deeply, and any other StarLayer error.

In [ ]:
def read(text):
    try:
        sparql_to_tree(text)
    except FormatError as error:
        print("malformed", error.kind, "at line", error.line)
    except ExecutionError as error:
        only_kind(error, "nesting")  # handle only this kind; any other kind is raised again
        print("too deep to read")
    except StarLayerError as error:
        print("another StarLayer error:", type(error).__name__)
    else:
        print("read")


read("SELECT ?s WHERE { ?s ?p ?o }")
read("not a query")
read("SELECT ?x WHERE " + "{ " * 3000 + "?x ?p ?o" + " }" * 3000)

read
malformed sparql at line 1
too deep to read


## 8. What changed from the earlier release

If your code caught one of the old errors, this is what to catch now.

| You used to catch | Now catch |
| --- | --- |
| `SyntaxError` from `parse()` | `FormatError`, kind `rdf` |
| `SyntaxError` or `SparqlSyntaxError` from a query or update text | `FormatError`, kind `sparql` |
| `RuntimeError` from `query()` or `update()` | `ExecutionError`, kind `query` |
| `SQEError`, `SQEValidationError`, `SPARQL12ConformanceError` | `FormatError`, kind `sparql`. A tree that breaks the `sqe:` shapes is no longer an error at parse time: `validate_sparql_tree()` reports it. |
| `ValueError` from `parse_term`, `to_python`, or an invalid IRI or language tag | `FormatError`, kind `term` (and `TypeError` for a value that is not text) |
| `ValueError` from `graph.items()` for a bad list | `FormatError`, kind `rdf` |
| `SRLParseError`, `SRLDecodeError`, `SRLEvalError` | `FormatError`, kind `srl`. A rule set is no longer checked when it is parsed: `validate_sparql_tree()` reports the problems and running the rule set raises. |
| `StratificationError` | `ExecutionError`, kind `rule` |
| `SRLImportError`, `SRLImportsNotSupportedError` | `ExecutionError`, kind `import` (or `FormatError`, kind `srl`, when the imported text is not a rule set) |
| `ManchesterSyntaxError` | `FormatError`, kind `manchester` |
| a graph Manchester Syntax could not write | `ExecutionError`, kind `unsupported` (or `FormatError`, kind `owl`, for RDF that is not valid OWL) |
| `InvalidTripleTermError`, `BasicEncodingError` | `FormatError`, kind `term` or `rdf` |
| `CanonicalizationComplexityError` | `ExecutionError`, kind `limit` |
| `EntailmentRewriteError`, `SqeTextTooDeepError` | `ExecutionError`, kind `unsupported` or `nesting` |
| `ReasoningTimeoutError`, `InconsistentOntologyError` | `ExecutionError`, kind `timeout` or `inconsistent` (`inference` when the reasoner could not run) |
| a missing OWL reasoner package as `RuntimeError` | `ImportError` |
| `NotImplementedError` for an unknown `entailment=` value | `ValueError` |
| `except ValueError` for a wrong shapes graph | `ShapeError` (it is no longer a `ValueError`) |
| the tuple `(conforms, report_graph, report_text)` from a validator | a `ValidationReport` with `.conforms`, `.report_graph` and `.report_text` |

The functions were renamed in the same release: `sparql_to_tree`, `tree_to_sparql` and `validate_sparql_tree` for SPARQL, updates and SRL, and `manchester_to_tree`, `tree_to_manchester`, `tree_to_owl` and `validate_manchester_tree` for Manchester Syntax.

## 9. Open questions

These are the judgment calls in the scheme. Each has a recommendation, and the alternative if you disagree.

1. **SHACL limits.** `max_iterations` and `max_triples` failures are `ExecutionError` kind `rule`, while the RDFC-1.0 size cap is kind `limit`. They have the same cause, a size cap. *Recommendation:* move the two SHACL limits to `limit`. *Cost:* about 12 assertions, guides 05.02 and 05.06.
2. **Term text passed as an argument.** An invalid IRI, language tag or CURIE given to `literal()`, `graph_name()` or `expand_curie()` is `FormatError` kind `term`, because the text usually comes from a person or a document. *Alternative:* `ValueError` for every argument. *Recommendation:* keep `FormatError`.
3. **Unknown entailment regime.** `infer(entailment="nonsense")` is a `ValueError`, because changing the argument fixes it. The old behaviour was `NotImplementedError`. *Alternative:* `ExecutionError` kind `unsupported`. *Recommendation:* keep `ValueError`.
4. **Where `ValueError` ends and `unsupported` starts.** A call refused by an object (`relevant_subgraph` given a `DESCRIBE`, writing a blank node that came from an endpoint) stays `ValueError`: you change what you pass. A construct inside a document that cannot be handled is `unsupported`. Some reviewers may prefer `unsupported` for the first group. *Recommendation:* keep the test "does changing the argument fix it?".
5. **A rule body element with no encoding** (`srl.py`) is `unsupported`, which follows the plan. `internal` is arguable, because the list of body elements is closed. *Recommendation:* keep `unsupported`.
6. **A call on a closed `Materialization`** is kind `rule`. No new kind for one place. *Alternative:* keep `ValueError`. *Recommendation:* keep `rule`.
7. **Impossible states** (a pretty-writer guard, an endpoint guard, ten "unhandled tag" sites in the Manchester code) are `ExecutionError` kind `internal`, so they read as a bug to report and not as an `assert`. *Recommendation:* keep.
8. **File and I/O errors are not wrapped.** A missing file is still `FileNotFoundError`, and Oxigraph's `OSError` from `serialize()` stays as it is. *Recommendation:* keep.
9. **Deep nesting is an `ExecutionError`, not a `FormatError`.** The input is valid but too deep to process. A caller who catches only `FormatError` for untrusted queries would miss it. *Recommendation:* keep, and document that untrusted input needs `except StarLayerError`.
10. **Kinds that could be merged:** `manchester` and `owl`, `rdf` and `term`, `import` (three places). *Recommendation:* keep them separate; a merged kind loses information.
11. **`EndpointError` has two bases** (`StarLayerError` and `OSError`), so callers who catch `OSError` for a network failure keep working. *Alternative:* `ExecutionError` kind `endpoint` with a `status`. *Recommendation:* keep.
12. **`context` is free text.** It is not a registry, so a reader can only match on the text. *Recommendation:* keep for now; add a registry only if callers start matching on it.
13. **`ShapeError` is no longer a `ValueError`.** This is the change most likely to break existing callers that wrote `except ValueError` around shape code. *Recommendation:* keep; it is listed in the table above.
14. **Housekeeping.** The generated API docs (`docs/generated/api-*.generated.md`) and the code graph still name removed classes and need regenerating. Two tests of the one-off guide renumbering tool (`tests/repo/test_renumber_guides.py`) fail, and the tool can be retired.